# Capstone: Who Gets the Deep Network?

A deployed classifier runs the same network on every input. A photo that any model
would get right costs exactly as much as one that is genuinely ambiguous.

Early exit breaks that. You attach a small classifier partway up the network, and
when it is confident enough you stop there and skip the rest. Averaged over a stream
of inputs, you spend less.

The usual way to describe this is as a saving: exit early, use less compute, lose a
little accuracy. That framing hides the more interesting question, because it compares
two different budgets. Fix the budget instead. Give every policy the same average cost
per image and let them differ only in how they spend it.

That is the question this project asks, and it is open before you measure it. At a
fixed average compute budget, does choosing which examples get the deep path beat
choosing arbitrarily?

Write down your answer now. You will check it later.

<div style="border-left:6px solid #A31F34;background:#fff5f6;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#A31F34;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Big picture</div>
<p style="margin:6px 0 0;">The control here is not "always run the full network". It is random allocation: the same budget, spent without looking at the example. That is what isolates the routing decision. A policy that beats always-deep has only shown that spending less costs accuracy, which nobody doubted. Every lab in this course handed you the intervention. Here you author it yourself.</p>
</div>

## At a glance

| | |
|---|---|
| Runtime | roughly 3 to 6 minutes on a free Colab GPU runtime, much of it the one-time CIFAR-10 download |
| Data | CIFAR-10, downloaded automatically, upsampled to 160x160 |
| Model | a frozen ImageNet ResNet-18 with one linear exit head per stage |
| You design | the routing score, the hypothesis, and the smallest difference worth reporting |
| You submit | seven values, R1 through R7, from the final cell |

## What is graded

Five course-page gates.

R1 through R3 check three analysis helpers on fixed inputs. Those inputs never
change, so the answers are definite and are the same for every capstone path.

R4 and R5 are your proposal: a design that satisfies the contract, and the compute it
will cost. You get both before running anything.

R6 and R7 are the execution record: a contract over the experiment you actually ran,
and the compute it actually spent.

Gates 4 and 5 ask you to read your own result. There is no single correct outcome.

Your comparison will land in one of three places, and all three are complete
findings: the effect is at least as large as the one you declared worth acting on, it
is at most that large, or this many seeds cannot tell those apart. Reporting the third
honestly scores as well as reporting the first.

## Setup

The first cell fetches the shared contract layer. It runs in Colab and on a local
machine, and it does nothing if the files are already present.

In [ ]:
# Colab bootstrap: fetch the shared capstone modules if they are not already here.
import hashlib
import os
import urllib.request

REPO = ("https://raw.githubusercontent.com/codey-m/deep_learning/main/"
        "final_project/helpers")
NEEDED = ["project_schema.py", "earlyexit.py", "earlyexit_adapter.py"]
# Digests of the exact module versions this notebook was built and tested against. A
# file that does not match is a stale copy from an earlier session or a truncated
# download, and either one would fail later in a way that looks like your mistake.
DIGESTS = {
    "earlyexit.py": "db5d7b9e65ad307643e4b463abe57f8eb1186eef9dd3a750be0a6d45a590fa7f",
    "earlyexit_adapter.py": "cfad85dd39b862440911f961de5b4ebe9690c8b77a8df3bef4aa26db4ffa9e14",
    "project_schema.py": "01a72e074e34ae3e571e05b7034d81ee71f40b79be67bc267c0658b5811dfa17",
}


def digest_of(path):
    with open(path, "rb") as handle:
        return hashlib.sha256(handle.read()).hexdigest()


for name in NEEDED:
    if not os.path.exists(name) or digest_of(name) != DIGESTS[name]:
        urllib.request.urlretrieve(f"{REPO}/{name}", name)
    if digest_of(name) != DIGESTS[name]:
        raise RuntimeError(
            f"{name} does not match the version this notebook was tested against. "
            f"Delete it and restart the runtime.")
print("contract layer ready:", ", ".join(NEEDED))

In [ ]:
import hashlib
import math
import statistics
import time

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

import earlyexit
import project_schema as schema
import earlyexit_adapter as adapter
from project_schema import Contrast, ProjectPlan, RunRecord

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu")

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

# QUICK_MODE is the graded default and is what the runtime estimate above assumes.
# Turning it off trains the heads longer and replicates over more seeds, which tightens
# the noise floor without changing the design.
QUICK_MODE = True
RESOLUTION = 160
TRAIN_PER_CLASS, EVAL_PER_CLASS = 300, 200
HEAD_EPOCHS = 30 if QUICK_MODE else 60
HEAD_BATCH = 256
SEEDS = tuple(7960 + i for i in range(10 if QUICK_MODE else 16))
METRIC = "accuracy"

print(f"device: {DEVICE}")
print(f"instrument self-check: {earlyexit.self_check()}")

### The network, split into stages

A ResNet-18 is a stem followed by four residual stages. Pool the activations after any
stage and a linear classifier can read them, which gives four possible exits without
touching the trunk.

The backbone stays frozen. Only the four exit heads are trained, and they are trained
once per seed and shared by every policy. A policy that trained its own heads would be
competing with a different model rather than a different rule, and the contract
rejects it.

In [ ]:
train_set = datasets.CIFAR10("data", train=True, transform=transforms.ToTensor(),
                             download=True)
test_set = datasets.CIFAR10("data", train=False, transform=transforms.ToTensor(),
                            download=True)
train_targets = torch.as_tensor(train_set.targets)
test_targets = torch.as_tensor(test_set.targets)

backbone = models.resnet18(
    weights=models.ResNet18_Weights.IMAGENET1K_V1).to(DEVICE).eval()
STEM, STAGES = earlyexit.stage_modules(backbone)


@torch.no_grad()
def features_for(dataset, indices):
    """Pooled features after every stage, in one pass over the images."""
    collected = [[] for _ in STAGES]
    labels = []
    for images, target in DataLoader(Subset(dataset, indices), batch_size=128,
                                     shuffle=False):
        images = nn.functional.interpolate(images, size=(RESOLUTION, RESOLUTION),
                                           mode="bilinear", align_corners=False)
        images = ((images - MEAN) / STD).to(DEVICE)
        for index, feature in enumerate(
                earlyexit.stage_features(STEM, STAGES, images, DEVICE)):
            collected[index].append(feature)
        labels.append(target)
    return [torch.cat(part) for part in collected], torch.cat(labels)


picker = torch.Generator().manual_seed(11)
train_idx, eval_idx = [], []
for class_id in range(10):
    pool = torch.where(train_targets == class_id)[0]
    train_idx += pool[torch.randperm(len(pool), generator=picker)][
        :TRAIN_PER_CLASS].tolist()
    pool = torch.where(test_targets == class_id)[0]
    eval_idx += pool[torch.randperm(len(pool), generator=picker)][
        :EVAL_PER_CLASS].tolist()

started = time.time()
train_features, train_labels = features_for(train_set, sorted(train_idx))
eval_features, eval_labels = features_for(test_set, sorted(eval_idx))

# The held-out set is split in two, and the halves hold different images. A policy may
# solve its threshold on the calibration half and is scored on the other. Solving it on
# the images it will be judged on would be an oracle operating point rather than a
# deployable rule, which is the mistake the abstention path is entirely about.
_perm = torch.randperm(len(eval_labels), generator=torch.Generator().manual_seed(5))
calib_pos, score_pos = _perm[:len(eval_labels) // 2], _perm[len(eval_labels) // 2:]
calib_features = [f[calib_pos] for f in eval_features]
score_features = [f[score_pos] for f in eval_features]
score_labels = eval_labels[score_pos]
eval_digest = hashlib.sha256(
    repr(score_labels.tolist()).encode("utf-8")).hexdigest()[:16]
print(f"features cached in {time.time() - started:.0f}s")
print(f"train {len(train_labels)}  calibrate {len(calib_pos)}  "
      f"score {len(score_pos)}  digest {eval_digest}")
print(f"feature width per exit: {[f.shape[1] for f in train_features]}")

### What a stage actually costs

Cost here is measured on the clock, not counted in layers. The two are not the same,
and the difference decides whether this whole idea is worth anything.

Run the cell and look at the first number. It is the cost of reaching exit 1, which
means the stem plus one residual stage. Compare it to the last number, the full
network.

In [ ]:
stage_costs = earlyexit.measure_stage_costs(STEM, STAGES, DEVICE, RESOLUTION)
target_cost = (stage_costs[-2] + stage_costs[-1]) / 2

print(f"cumulative ms per image to reach each exit: "
      f"{[round(c, 4) for c in stage_costs]}")
print(f"exiting at stage 1 still costs {stage_costs[0] / stage_costs[-1]:.0%} of the "
      f"full forward pass")
print(f"\nthe budget every policy must hit: {target_cost:.4f} ms per image")
print(f"that is the average of the two deepest exits, so no policy can reach it by "
      f"sending everything to one stage")

<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">That first fraction is the honest limit on this project. The stem runs before any exit can be taken, so it is charged to every example no matter how early you stop. The whole range a routing policy can move within is narrower than a layer count suggests, and any claim you make about savings has to live inside it.</p>
</div>

## Matched cost, and why it is the whole design

Three policies will be compared. Each one decides, per example, where to stop.

The control is random allocation. It hits the budget by sending a fixed fraction of
examples to the deeper exit, chosen at random, without looking at the image at all.

The supplied baseline exits as soon as the max softmax probability clears a threshold.

The third is yours.

All three are calibrated to the same average cost. The threshold is not chosen, it is
solved for: the harness searches for the value whose realized cost lands on the
budget. That is what makes the comparison a question about allocation rather than a
question about who was allowed to spend more.

The threshold is solved on the calibration half and applied to the scoring half, and
those halves hold different images. Solving it on the images the policy will be judged
on would give every policy a perfect operating point it could not have had in
deployment, and would quietly reward a score whose distribution does not transfer.
Because the threshold is fitted elsewhere, the realized cost lands near the budget
rather than exactly on it, and the contract allows that small drift.

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 1</div>
<div style="font-weight:700;font-size:1.2rem;">Scope a run to a compute budget</div>
</div>

Every lab so far handed you a compute budget. Estimating one is a skill in itself, and
it is the difference between an experiment that finishes inside a Colab session and
one that dies partway through with nothing to show.

Use the cheapest estimate that tracks real cost: the work of a training run scales
with the number of parameters receiving gradients, multiplied by the number of
examples pushed through. The backbone is frozen, so none of its parameters count here.
Only the exit heads are trained. Return the count in billions, so the numbers stay
readable:

$$\text{budget units} = \frac{\text{trainable parameters} \times \text{steps} \times \text{batch size}}{10^9}$$

<div style="border-left:6px solid #1D4ED8;background:#eff6ff;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#1D4ED8;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Intuition</div>
<p style="margin:6px 0 0;">This is a proxy, not a stopwatch. It will not predict wall-clock seconds on a particular GPU, and it is not meant to. What it does is rank designs against each other and catch the one that is a hundred times larger than you thought. The millisecond costs measured above are a separate instrument, for a separate question.</p>
</div>

In [ ]:
# STUDENT TASK 1: estimate the cost of a run in budget units.
def compute_budget(trainable_parameters, steps, batch_size):
    """Trainable parameters times examples processed, in billions."""
    # TODO: return trainable_parameters * steps * batch_size, expressed in billions.
    return 0.0

In [ ]:
# Probe 1 (fixed input, definite answer): 35,000 parameters, 250 steps, batch size 50.
probe_budget_value = round(compute_budget(35_000, 250, 50), 4)
budget_probe_contract = abs(probe_budget_value - 0.4375) < 1e-4
print(f"R1 probe: {probe_budget_value} budget units "
      f"({'matches' if budget_probe_contract else 'does not match'} the expected 0.4375)")

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 2</div>
<div style="font-weight:700;font-size:1.2rem;">Compare two conditions the paired way</div>
</div>

You will run each policy under ten random seeds. Seeds change how the exit heads are
initialized and the order they see their training batches, so the same policy does not
give the same accuracy twice.

The naive comparison takes the mean of one condition and subtracts the mean of the
other. The paired comparison takes the difference within each seed first, then
averages those differences.

With the same seeds in both conditions these two give the identical mean. What changes
is the uncertainty around it. Seed 7963 might produce weak exit heads for every policy
at once, and when you difference within that seed, its weakness cancels. Differencing
the group means leaves it in.

How much pairing buys depends on how much the two conditions actually share, so it
is not a fixed number and is not worth taking on faith. When you read the result you
will see both floors, the paired one and the one you would have got by differencing
the group means, and you can judge the size of the difference on your own run.

In [ ]:
# STUDENT TASK 2: the mean within-seed difference between two conditions.
def paired_difference(treatment_by_seed, reference_by_seed):
    """Both arguments map seed -> metric value. Use only the seeds present in both."""
    seeds = sorted(set(treatment_by_seed) & set(reference_by_seed))
    # TODO: build the list of within-seed differences (treatment minus reference),
    # then return their mean.
    return 0.0

In [ ]:
# Probe 2 (fixed input, definite answer): three seeds, two conditions.
PROBE_TREATMENT = {1: 0.30, 2: 0.28, 3: 0.26}
PROBE_REFERENCE = {1: 0.34, 2: 0.33, 3: 0.29}
probe_paired_value = round(paired_difference(PROBE_TREATMENT, PROBE_REFERENCE), 4)
paired_probe_contract = abs(probe_paired_value - (-0.04)) < 1e-4
print(f"R2 probe: {probe_paired_value} "
      f"({'matches' if paired_probe_contract else 'does not match'} the expected -0.04)")

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 3</div>
<div style="font-weight:700;font-size:1.2rem;">Decide when a difference is big enough to believe</div>
</div>

A mean difference on its own settles nothing. Ten seeds is a small sample, and a
difference of one accuracy point means one thing when the seed-to-seed spread is a
tenth of a point and another thing entirely when the spread is five points.

The floor is the half-width of a t-interval on the paired differences:

$$\text{floor} = t_{0.975,\, n-1} \cdot \frac{s}{\sqrt{n}}$$

where $s$ is the sample standard deviation of the within-seed differences and $n$ is
how many you have. A difference is resolved when its magnitude exceeds its own floor.
Anything smaller is inside the noise your own seeds produce, and you cannot tell it
apart from zero.

The critical value is supplied. You combine the pieces.

<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">Resolving a difference and caring about one are separate questions, and the verdict below combines them into a single call. If the whole interval sits beyond your SESOI, the effect is at least the size you said you would act on. If the whole interval sits inside it, the effect is at most that size, which is a real result and not a failure. If the interval straddles your SESOI, the honest answer is that this many seeds cannot tell, and the fix is more seeds rather than a softer claim.</p>
</div>

In [ ]:
# STUDENT TASK 3: the paired noise floor for a list of within-seed differences.
def resolution_floor(differences, critical_value):
    """Half-width of the t-interval around the mean of ``differences``."""
    count = len(differences)
    # TODO: return critical_value * (sample standard deviation) / sqrt(count).
    return 0.0

In [ ]:
# Probe 3 (fixed input, definite answer): the same three seeds as probe 2.
PROBE_DIFFERENCES = [PROBE_TREATMENT[s] - PROBE_REFERENCE[s] for s in (1, 2, 3)]
probe_floor_value = round(
    resolution_floor(PROBE_DIFFERENCES, schema._critical(len(PROBE_DIFFERENCES))), 4)
floor_probe_contract = abs(probe_floor_value - 0.0248) < 1e-4
print(f"R3 probe: {probe_floor_value} "
      f"({'matches' if floor_probe_contract else 'does not match'} the expected 0.0248)")
print(f"the probe difference of {probe_paired_value} is "
      f"{'resolved' if abs(probe_paired_value) > probe_floor_value else 'inside the noise'}")

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 4</div>
<div style="font-weight:700;font-size:1.2rem;">Design your routing score</div>
</div>

This is the part that is yours. Write a function that takes the class probabilities at
one exit and returns one number per example: how confident this exit is about this
example. Higher means more confident, so the example exits sooner.

You never choose a threshold. The harness solves for the threshold that puts your
score on the budget, using the calibration half, and then applies it to images your
score has not been tuned against. That is what keeps every policy spending the same
amount without letting any of them see the answer sheet. Your job is only to decide
what signal the decision reads.

Your score does not have to be a probability, or lie in any particular range.
Calibration searches over whatever range your score occupies.

<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">Your function receives probabilities and nothing else. It cannot see the label, and the contract inspects the routing policy's arguments to confirm no condition was given one. A rule that routes by knowing the answer is not a routing rule.</p>
</div>

<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Signals worth considering</summary>
<div style="margin-top:8px;"><ul><li><b>Margin.</b> The gap between the top two probabilities. A confident-looking maximum can still be a near tie between two classes, and those are the examples worth spending depth on.</li><li><b>Negative entropy.</b> Uses the whole distribution rather than the top one or two entries, so mass spread thinly over eight classes reads differently from mass split between two.</li><li><b>Top-k mass.</b> How much probability the top three classes hold together. Cheap, and insensitive to how the leader and runner-up divide it.</li><li><b>Agreement with the previous exit.</b> You have the earlier exits' outputs available, so a score can ask whether the shallow exits already agree with each other. Stability across depth is a different signal from confidence at one depth.</li></ul></div>
</details>

In [ ]:
# STUDENT TASK 4: author your routing score.
def learner_score(probabilities):
    """probabilities: a (N, 10) tensor of class probabilities at one exit.

    Return a (N,) tensor. Higher means more confident, so the example exits here.
    """
    # TODO: replace this with your own signal. Returning the top probability would
    # reproduce the supplied baseline exactly, which the design contract rejects.
    return probabilities.max(dim=1).values


MY_SCORE_REASON = ""  # TODO: one sentence on what your score notices that max softmax misses.

In [ ]:
def scores_at_each_exit(heads, features, score_function):
    """Apply a scoring function at every exit, with that exit's own prediction."""
    out = []
    with torch.no_grad():
        for head, feature in zip(heads, features):
            probability = head(feature).softmax(dim=1)
            out.append((score_function(probability), probability.argmax(dim=1)))
    return out


def max_softmax_score(probabilities):
    """The supplied baseline: the single largest class probability."""
    return probabilities.max(dim=1).values


# Does the score behave like a score? Checked on a fixed probe, before anything trains.
_probe = torch.softmax(torch.randn(256, 10, generator=torch.Generator().manual_seed(3)),
                       dim=1)
_learner_out = learner_score(_probe)
score_is_well_formed = (
    isinstance(_learner_out, torch.Tensor)
    and _learner_out.shape == (len(_probe),)
    and bool(torch.isfinite(_learner_out).all()))
score_is_novel = not (
    score_is_well_formed
    and torch.allclose(_learner_out.float(), max_softmax_score(_probe), atol=1e-6))

print(f"well formed: {score_is_well_formed}   differs from max softmax: "
      f"{score_is_novel}")
if score_is_well_formed:
    print(f"your score spans [{_learner_out.min():.4f}, {_learner_out.max():.4f}] "
          f"on the probe; calibration searches this range")

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 5</div>
<div style="font-weight:700;font-size:1.2rem;">Declare your prediction before you measure</div>
</div>

Four declarations, all made before a single exit head is trained.

**The hypothesis.** What you expect your score to do to held-out accuracy at the fixed
budget, and the mechanism you think would cause it.

**The required contrast.** Which comparison your project stands on. Your score against
random allocation asks whether looking at the example helps at all. Your score against
max softmax asks the harder question, whether looking at it your way helps more than
the obvious way.

**The smallest effect worth caring about.** In accuracy points. Declaring 0.010 says a
one-point gain would change what you would deploy and anything smaller would not, even
if you could resolve it.

**The budget.** What the whole experiment will cost, from Task 1.

<div style="border-left:6px solid #A31F34;background:#fff5f6;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#A31F34;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Big picture</div>
<p style="margin:6px 0 0;">Declaring the effect size you care about before measuring is what separates an experiment from a search. If you fix it afterwards, you will fix it wherever your result happened to land, and you will have learned nothing you did not already assume.</p>
</div>

In [ ]:
# STUDENT TASK 5: declare the design.
MY_HYPOTHESIS = ""  # TODO: what you expect, and the mechanism you think causes it.

# TODO: which comparison does your project stand on? treatment is always
# "learner_score"; reference is "random_allocation" or "max_softmax"; direction is
# "greater" (your score raises accuracy) or "less" (it lowers it).
REQUIRED_CONTRAST = Contrast("learner_vs_random", "learner_score", "random_allocation",
                             required=True, direction="greater")

# TODO: the smallest change in accuracy you would act on, in accuracy points.
SESOI_ABSOLUTE = 0.010

In [ ]:
CONDITIONS = ("random_allocation", "max_softmax", "learner_score")
SCORE_FOR = {"random_allocation": max_softmax_score,
             "max_softmax": max_softmax_score,
             "learner_score": learner_score}


# The three policies. Their signatures are part of the contract: none may accept
# labels, and the adapter inspects them to confirm it.
def policy_random(per_exit, stage_costs, target_cost, seed):
    """The control: the same budget, spent without looking at the example."""
    count = len(per_exit[0][0])
    costs = torch.tensor(stage_costs, dtype=torch.float)
    upper = int((costs >= target_cost).nonzero()[0])
    lower = max(0, upper - 1)
    share = (0.0 if costs[upper] == costs[lower]
             else float((target_cost - costs[lower]) / (costs[upper] - costs[lower])))
    draw = torch.rand(count, generator=torch.Generator().manual_seed(seed + 31337))
    return torch.where(draw < share, torch.full((count,), upper),
                       torch.full((count,), lower))


def policy_threshold(calibration_exit, per_exit, stage_costs, target_cost):
    """Solve the threshold on the calibration half, then apply it to the scoring half."""
    thresholds = earlyexit.calibrate_to_budget(calibration_exit, stage_costs,
                                               target_cost)
    return earlyexit.apply_policy(per_exit, thresholds)[0]


POLICIES = {"random_allocation": policy_random,
            "max_softmax": policy_threshold,
            "learner_score": policy_threshold}

CONTRASTS = (
    REQUIRED_CONTRAST,
    Contrast("max_softmax_vs_random", "max_softmax", "random_allocation",
             direction="greater"),
    Contrast("learner_vs_max_softmax", "learner_score", "max_softmax",
             direction="greater"),
)
configs = {c: {"exit_policy": c} for c in CONDITIONS}

head_parameters = sum(f.shape[1] * 10 + 10 for f in train_features)
steps_per_seed = HEAD_EPOCHS * math.ceil(len(train_labels) / HEAD_BATCH)
projected_budget_units = round(
    compute_budget(head_parameters, steps_per_seed, HEAD_BATCH) * len(SEEDS), 4)

plan = ProjectPlan(
    path="early_exit",
    question="At a fixed average compute budget, does choosing which examples get the "
             "deep path beat choosing arbitrarily?",
    hypothesis=MY_HYPOTHESIS,
    control="random_allocation",
    intervention="exit_policy",
    declared_change="exit_policy",
    conditions=CONDITIONS,
    evaluation_slices=(adapter.EVAL_SLICE, adapter.COST_SLICE),
    seeds=SEEDS,
    compute_budget=projected_budget_units,
    decisions=(f"target_cost={target_cost:.4f}", f"sesoi={SESOI_ABSOLUTE}"),
    condition_kind="categorical",
    contrasts=CONTRASTS,
    # One fixed dataset for every seed. Replication is over head initialization and
    # batch order, not over which images were drawn.
    seed_varies="training_only",
)

BUDGET_MIN, BUDGET_MAX = 0.0, 10000.0

# The generic proposal contract, run now rather than after the experiment. This is the
# same check the execution gate applies to the plan, so a malformed design fails here
# in a second instead of after the run.
proposal = schema.ContractResult()
schema.check_plan(plan, budget_min=BUDGET_MIN, budget_max=BUDGET_MAX, result=proposal)

design_contract = int(
    score_is_well_formed
    and score_is_novel
    and len(MY_HYPOTHESIS.split()) >= 12
    and len(MY_SCORE_REASON.split()) >= 6
    and REQUIRED_CONTRAST.required
    and REQUIRED_CONTRAST.treatment == "learner_score"
    and REQUIRED_CONTRAST.reference in ("random_allocation", "max_softmax")
    and REQUIRED_CONTRAST.direction in ("less", "greater")
    and 0.0 < SESOI_ABSOLUTE <= 0.25
    and proposal.passed
    and bool(plan.freeze()))

print(f"trainable head parameters: {head_parameters:,}   steps per seed: "
      f"{steps_per_seed}")
print(f"R4 design contract: {design_contract}")
if not proposal.passed:
    print(proposal.report())
print(f"R5 projected budget: {projected_budget_units} units for "
      f"{len(CONDITIONS)} policies x {len(SEEDS)} seeds")
print(f"plan digest: {plan.freeze()}")

# Bind every measurement to the design and to the code that produced it. The bytecode
# and constants of your own function go into the digest alongside the frozen plan, so a
# result table can be traced to the exact method that made it and a plan edited after
# the run stops matching its own rows. A notebook can always be re-run from the top,
# so this detects a changed design rather than preventing one.
provenance = hashlib.sha256(
    plan.freeze().encode("utf-8")
    + learner_score.__code__.co_code
    + repr(learner_score.__code__.co_consts).encode("utf-8")).hexdigest()[:16]
print(f"run provenance: {provenance}")

<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">R4 must read 1 before you go on. If it does not, one of the declarations above is still a placeholder, or your score still returns the top probability. Running the experiment on a design that fails its own contract wastes the compute and the result will not be gradeable.</p>
</div>

## Run the experiment

Exit heads are trained once per seed and shared by all three policies, so the
conditions differ only in the routing rule.

In [ ]:
def train_heads(features, labels, seed):
    """One linear exit head per stage. Returns the heads and the budget they spent."""
    torch.manual_seed(seed)
    heads, spent = [], 0.0
    shuffler = torch.Generator().manual_seed(seed * 31)
    for feature in features:
        head = nn.Linear(feature.shape[1], 10).to(DEVICE)
        parameters = sum(p.numel() for p in head.parameters())
        optimizer = torch.optim.Adam(head.parameters(), lr=1e-2, weight_decay=1e-4)
        inputs, targets = feature.to(DEVICE), labels.to(DEVICE)
        for _ in range(HEAD_EPOCHS):
            order = torch.randperm(len(inputs), generator=shuffler).to(DEVICE)
            for start in range(0, len(order), HEAD_BATCH):
                pick = order[start:start + HEAD_BATCH]
                optimizer.zero_grad()
                nn.functional.cross_entropy(head(inputs[pick]),
                                            targets[pick]).backward()
                optimizer.step()
                # The projection assumed every batch was full. This counts the examples
                # that actually went through, so R7 is measured rather than restated.
                spent += compute_budget(parameters, len(pick), 1)
        heads.append(head.cpu())
    return heads, spent


def head_digest(heads):
    return hashlib.sha256(
        repr([h.weight.detach().cpu().numpy().tobytes() for h in heads])
        .encode("utf-8")).hexdigest()[:16]


started = time.time()
records, realized_costs, depths_by_condition = [], {}, {}
method_seconds = {}
head_digests, eval_digests = {}, {}
measured_budget_units = 0.0

for seed in SEEDS:
    heads, spent = train_heads(train_features, train_labels, seed)
    measured_budget_units += spent
    shared_heads = head_digest(heads)
    for condition in CONDITIONS:
        # R5 and R7 price the supplied model. They cannot price learner-authored
        # code, so it is timed instead. Diagnostic only: a wall clock reading is
        # not comparable between one machine and another.
        scoring_started = time.perf_counter()
        calibration_exit = scores_at_each_exit(heads, calib_features,
                                               SCORE_FOR[condition])
        per_exit = scores_at_each_exit(heads, score_features, SCORE_FOR[condition])
        scoring_seconds = time.perf_counter() - scoring_started
        method_seconds[condition] = method_seconds.get(condition, 0.0) + scoring_seconds
        if condition == "random_allocation":
            depth = POLICIES[condition](per_exit, stage_costs, target_cost, seed)
        else:
            depth = POLICIES[condition](calibration_exit, per_exit, stage_costs,
                                        target_cost)
        prediction = torch.stack([p for _, p in per_exit], dim=1)[
            torch.arange(len(depth)), depth]

        realized_costs[condition] = earlyexit.realized_cost(depth, stage_costs)
        depths_by_condition[condition] = depth.tolist()
        head_digests[condition] = shared_heads
        eval_digests[condition] = eval_digest
        for name, value, slice_name in (
                (METRIC, earlyexit.accuracy(prediction, score_labels),
                 adapter.EVAL_SLICE),
                ("realized_cost", realized_costs[condition], adapter.COST_SLICE)):
            records.append(RunRecord(
                condition=condition, seed=seed, split_hash="fixed_split",
                config_hash=schema.config_hash(configs[condition]),
                training_steps=HEAD_EPOCHS, runtime_seconds=scoring_seconds, provenance=provenance,
                metric_name=name, evaluation_slice=slice_name, value=value))
    print(f"  seed {seed} done ({time.time() - started:.0f}s elapsed)", flush=True)

measured_budget_units = round(measured_budget_units, 4)
print(f"\n{len(records)} records in {time.time() - started:.0f}s")
print(f"budget projected {projected_budget_units}, measured {measured_budget_units}")

### Does the saving reach the clock?

A cost model that says a policy is cheaper is a claim about the hardware, and claims
about hardware should be checked against the hardware.

The cell below re-runs each policy's actual depth distribution on a genuinely
truncated batch and times it. A ratio near 1.0 means the predicted saving appears on
the clock. A ratio well above 1.0 means the saving existed only in the arithmetic,
which is the failure mode that makes published early-exit results untrustworthy.

In [ ]:
verification = {}
for condition, depth in depths_by_condition.items():
    counts = [sum(1 for d in depth if d == index) for index in range(len(STAGES))]
    verification[condition] = earlyexit.verify_savings_materialise(
        STEM, STAGES, counts, stage_costs, DEVICE, RESOLUTION)
    report_row = verification[condition]
    print(f"{condition:<20} predicted {report_row['predicted_ms']:.4f} ms  "
          f"measured {report_row['measured_ms']:.4f} ms  "
          f"ratio {report_row['ratio']:.2f}")

## Read the result

Look at the paired column, not the accuracy column. Two accuracies that look far apart
can sit inside a noise floor that is wider still, and two that look identical can be
cleanly separated once the shared seed variation is differenced away.

In [ ]:
values = schema.paired_values(records, METRIC, adapter.EVAL_SLICE)
report = schema.contrast_report(records, METRIC, adapter.EVAL_SLICE, CONTRASTS,
                                sesoi=SESOI_ABSOLUTE)
means = {c: statistics.mean(values[c].values()) for c in CONDITIONS}

print(f"{'policy':<20} {'accuracy':>10} {'sd':>8} {'cost':>9} {'clock':>8}")
for condition in CONDITIONS:
    column = [values[condition][s] for s in SEEDS]
    print(f"{condition:<20} {means[condition]:>10.4f} "
          f"{statistics.stdev(column):>8.4f} {realized_costs[condition]:>9.4f} "
          f"{verification[condition]['ratio']:>8.2f}")

print()
print(schema.format_contrasts(report, label="held-out accuracy at matched cost"))

# Your own helpers, applied to your own experiment. They must agree with the report.
required_row = next(row for row in report["rows"]
                    if row["name"] == REQUIRED_CONTRAST.name)
own_differences = [values[REQUIRED_CONTRAST.treatment][s]
                   - values[REQUIRED_CONTRAST.reference][s] for s in SEEDS]
own_mean = paired_difference(values[REQUIRED_CONTRAST.treatment],
                             values[REQUIRED_CONTRAST.reference])
own_floor = resolution_floor(own_differences, schema._critical(len(own_differences)))
helpers_agree = (abs(own_mean - required_row["mean"]) < 1e-9
                 and abs(own_floor - required_row["floor"]) < 1e-9)
print(f"\nyour helpers reproduce the report: {helpers_agree}")

# What pairing bought on this run: the same contrast judged by differencing the group
# means instead of within each seed. The factor depends on how much the two conditions
# share, so it is measured here rather than asserted.
_treat = [values[REQUIRED_CONTRAST.treatment][s] for s in SEEDS]
_ref = [values[REQUIRED_CONTRAST.reference][s] for s in SEEDS]
unpaired_floor = schema._critical(len(SEEDS)) * math.sqrt(
    (statistics.stdev(_treat) ** 2 + statistics.stdev(_ref) ** 2) / len(SEEDS))
if own_floor > 0:
    print(f"floor on your required contrast: {unpaired_floor:.4f} unpaired against "
          f"{own_floor:.4f} paired ({unpaired_floor / own_floor:.1f}x)")
    print("  pairing only buys something where the two conditions rise and fall "
          "together across seeds. A factor near 1.0 means yours barely do, which is a "
          "fact about the conditions rather than a mistake, and is worth a sentence in "
          "your caveat.")
else:
    print("floor comparison skipped: your resolution_floor returned 0, so Task 3 is "
          "not finished. Everything below this point depends on it.")

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(15, 4))

# A slope plot, not a boxplot. Every grey line is one seed carried across the
# conditions, so the pairing the whole analysis relies on is visible: what matters is
# whether the lines tilt the same way, not how wide the spread is.
for seed in SEEDS:
    axes[0].plot(range(len(CONDITIONS)), [values[c][seed] for c in CONDITIONS],
                 marker="o", ms=3, lw=0.8, alpha=0.45, color="#6b7280")
axes[0].plot(range(len(CONDITIONS)), [means[c] for c in CONDITIONS],
             marker="o", ms=8, lw=2.5, color="#A31F34", label="mean")
axes[0].set_xticks(range(len(CONDITIONS)))
axes[0].set_xticklabels([c.replace("_", "\n") for c in CONDITIONS], fontsize=8)
axes[0].set_ylabel("held-out accuracy")
axes[0].set_title("each seed, at matched cost")
axes[0].legend(fontsize=8)

width = 0.27
for offset, condition in zip((-width, 0.0, width), CONDITIONS):
    counts = [sum(1 for d in depths_by_condition[condition] if d == index)
              for index in range(len(STAGES))]
    axes[1].bar([i + offset for i in range(len(STAGES))], counts, width,
                label=condition.replace("_", " "))
axes[1].set_xticks(range(len(STAGES)))
axes[1].set_xticklabels([f"exit {i + 1}" for i in range(len(STAGES))])
axes[1].set_ylabel("examples")
axes[1].set_title("where each policy stops")
axes[1].legend(fontsize=8)

names = [row["name"] for row in report["rows"]]
axes[2].barh(names, [row["mean"] for row in report["rows"]], color="#A31F34")
for index, row in enumerate(report["rows"]):
    axes[2].plot([-row["floor"], row["floor"]], [index, index], color="#111827", lw=2)
axes[2].axvline(0, color="#6b7280", lw=1)
axes[2].axvline(SESOI_ABSOLUTE, color="#1D4ED8", ls="--", lw=1)
axes[2].tick_params(labelsize=8)
axes[2].set_xlabel("paired difference (bar), floor (black), SESOI (blue)")
axes[2].set_title("is the difference bigger than the noise?")
plt.tight_layout()
plt.show()

<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 6</div>
<div style="font-weight:700;font-size:1.2rem;">Write the record</div>
</div>

Six fields. The last two are the ones that decide whether this is science.

`claim` and `evidence` say what you found and the numbers behind it. `caveat` says
what your design could not control. `not_supported` says what a reader might
reasonably conclude from your result that your result does not actually show.
`next_experiment` names the single change you would make next.

Three of these are checked for form, not for quality, and the checks exist because
each names a way the record gets filled in without being thought about. `caveat` has
to name something the design held constant, because that is the boundary of what your
result covers. `resolved_comparisons` has to use the verdicts the analysis actually
returned, so it is written from the output rather than from memory. And
`not_supported` has to say something your claim does not, because pasting the claim
back is not a limitation.

Before you write anything, look back at the prediction you made at the top of this
notebook. If the result went the other way, say so: `claim` reports what happened, and
`not_supported` is not the place to quietly rewrite what you expected.

Write `not_supported` honestly. Beating random allocation shows that routing on
content helps, not that your particular signal is the reason. If your score did not
separate from max softmax, say so as a result: at this budget, on this backbone, the
choice of confidence signal did not matter by more than your declared SESOI, while the
choice to look at the example at all mattered a great deal. That is a sharper finding
than a win would have been.

In [ ]:
# STUDENT TASK 6: the experiment record. Every field must be non-empty.
learner_record = {
    "claim": "",             # TODO: what you found, in one sentence, with the numbers.
    "evidence": "",          # TODO: the measurements that support the claim.
    "resolved_comparisons": "",  # TODO: which contrasts cleared their floor, and by how much.
    "caveat": "",            # TODO: what your design could not control.
    "not_supported": "",     # TODO: what your result does NOT show.
    "next_experiment": "",   # TODO: the single change you would make next.
}

## The contract over what you ran

The checks below are the ones a research group would run before believing its own
result. Two of them decide whether the comparison means anything at all.

Realized cost must match the budget in every condition, or a policy that looks better
was simply allowed to spend more. And the predicted saving must appear on the clock,
because a policy that does not change execution does not change latency no matter what
the cost model says.

In [ ]:
result = adapter.run_all_checks(
    plan=plan, records=records, configs=configs, policies=POLICIES,
    realized_costs=realized_costs, target_cost=target_cost,
    verification=verification, head_digests=head_digests,
    depths=depths_by_condition, eval_digests=eval_digests,
    stage_count=len(STAGES), expected_examples=len(score_labels),
    record=learner_record, budget_min=BUDGET_MIN, budget_max=BUDGET_MAX, metric=METRIC,
    metrics=("realized_cost",),
    provenance=provenance, measured_budget=measured_budget_units)

execution_contract = int(
    result.passed
    and helpers_agree
    and abs(measured_budget_units - projected_budget_units)
    <= 0.10 * projected_budget_units)

print(f"contract passed: {result.passed}")
print(result.report())
print(f"\nR6 execution contract: {execution_contract}")

## Report values

Run the cell below once every task is complete and the experiment has finished.

R1 through R3 are the readiness values for Gate 1. R4 and R5 are the proposal for Gate
2, and you had both before the experiment ran. R6 and R7 are the execution record for
Gate 3. Gates 4 and 5 ask how you read your own result and what you would run next.

In [ ]:
probe_budget = probe_budget_value if budget_probe_contract else -1.0
probe_paired = probe_paired_value if paired_probe_contract else -1.0
probe_floor = probe_floor_value if floor_probe_contract else -1.0

report_values = {
    "R1: compute budget on the fixed probe": probe_budget,
    "R2: paired difference on the fixed probe": probe_paired,
    "R3: resolution floor on the fixed probe": probe_floor,
    "R4: design contract": design_contract,
    "R5: projected budget units for your design": projected_budget_units,
    "R6: execution contract": execution_contract,
    "R7: measured budget units your experiment spent": measured_budget_units,
}
assert abs(probe_budget - 0.4375) < 1e-4
assert abs(probe_paired - (-0.04)) < 1e-4
assert abs(probe_floor - 0.0248) < 1e-4
assert design_contract == 1
assert execution_contract == 1

print("CAPSTONE REPORT VALUES")
for label, value in report_values.items():
    print(f"{label}: {value}")